In [1]:
# !pip install trl
# !pip install unsloth
# !pip install bitsandbytes

In [1]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

project_root = "/content/drive/MyDrive/healthcare-ai-assistant"

DATA_DIR = os.path.join(project_root, "data")
REPORT_DIR = os.path.join(project_root, "reports")
MODEL_DIR = os.path.join(project_root, "saved_models")
NOTEBOOK_DIR = os.path.join(project_root, "notebooks")

for path in [DATA_DIR, REPORT_DIR, MODEL_DIR, NOTEBOOK_DIR]:
    os.makedirs(path, exist_ok=True)

print("✅ Project paths ready")

Mounted at /content/drive
✅ Project paths ready


### Imports

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from transformers import logging
logging.set_verbosity_error()

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
import torch
import json
import pandas as pd

max_seq_length = 2048
load_in_4bit = True

print("✅ Setup ready")

✅ Setup ready


### DPO/ORPO Preference Alignment

### Create 50 unique preference examples




In [3]:
preference_examples = [
    {
        "prompt": "What are the main symptoms of Type 2 diabetes?",
        "chosen": "Common symptoms of Type 2 diabetes include increased thirst, frequent urination, fatigue, blurred vision, slow wound healing, and increased hunger. A healthcare provider can confirm the diagnosis with blood glucose or A1C testing.",
        "rejected": "Diabetes just makes people tired sometimes."
    },
    {
        "prompt": "How can hospital-acquired infections be prevented?",
        "chosen": "Hospital-acquired infections can be reduced through strict hand hygiene, proper use of PPE, sterilization of equipment, environmental cleaning, antimicrobial stewardship, and infection prevention bundles.",
        "rejected": "Hospitals should just be cleaner."
    },
    {
        "prompt": "What does cancer metastasis mean?",
        "chosen": "Metastasis means cancer cells have spread from the original tumor to other parts of the body through the blood or lymphatic system. Common sites include the lungs, liver, bones, and brain.",
        "rejected": "It means cancer moves around."
    },
    {
        "prompt": "What are common cardiovascular diseases as people get older?",
        "chosen": "Common cardiovascular diseases with aging include coronary artery disease, heart failure, atrial fibrillation, stroke, and peripheral artery disease. Risk increases with hypertension, diabetes, smoking, and high cholesterol.",
        "rejected": "Older people usually get heart problems."
    },
    {
        "prompt": "What are common causes and symptoms of liver disease?",
        "chosen": "Common causes include viral hepatitis, alcohol-related liver disease, non-alcoholic fatty liver disease, autoimmune disease, and medication toxicity. Symptoms may include fatigue, jaundice, abdominal swelling, dark urine, and nausea.",
        "rejected": "Liver disease happens when the liver is bad."
    },
    {
        "prompt": "How should hypertension be managed?",
        "chosen": "Hypertension is managed with lifestyle changes such as reducing salt intake, regular exercise, weight control, limiting alcohol, and medications when needed. Regular blood pressure monitoring is important.",
        "rejected": "Just relax and drink water."
    },
    {
        "prompt": "What is the difference between Type 1 and Type 2 diabetes?",
        "chosen": "Type 1 diabetes is usually autoimmune and results from little or no insulin production. Type 2 diabetes is usually related to insulin resistance and relative insulin deficiency. Both require monitoring and medical management.",
        "rejected": "Type 1 happens to kids and Type 2 happens to adults."
    },
    {
        "prompt": "When should someone get a flu vaccine?",
        "chosen": "Most people should receive a flu vaccine every year, ideally before flu season. It is especially important for older adults, pregnant people, young children, and individuals with chronic medical conditions.",
        "rejected": "Only get it if you feel sick."
    },
    {
        "prompt": "What are warning signs of a heart attack?",
        "chosen": "Warning signs may include chest pain or pressure, shortness of breath, pain radiating to the arm, jaw, neck, or back, sweating, nausea, dizziness, and unusual fatigue. Emergency care should be sought immediately.",
        "rejected": "A heart attack is only chest pain."
    },
    {
        "prompt": "How can fatty liver disease be prevented?",
        "chosen": "Prevention includes maintaining a healthy weight, exercising regularly, limiting alcohol, eating a balanced diet, controlling diabetes and cholesterol, and avoiding unnecessary liver-toxic medications.",
        "rejected": "Do not eat fat."
    },
    {
        "prompt": "What is asthma?",
        "chosen": "Asthma is a chronic inflammatory airway disease that causes wheezing, coughing, chest tightness, and shortness of breath. It is often managed with inhalers and trigger avoidance.",
        "rejected": "Asthma is when breathing is weird."
    },
    {
        "prompt": "What triggers asthma symptoms?",
        "chosen": "Asthma symptoms can be triggered by allergens, respiratory infections, cold air, exercise, smoke, air pollution, and certain workplace exposures.",
        "rejected": "Asthma happens randomly."
    },
    {
        "prompt": "Why is medication reconciliation important?",
        "chosen": "Medication reconciliation helps prevent medication errors by comparing a patient's current medicines with new orders during admission, transfer, or discharge.",
        "rejected": "It is just checking pills."
    },
    {
        "prompt": "What is chronic kidney disease?",
        "chosen": "Chronic kidney disease is a long-term decline in kidney function. It is often monitored using estimated glomerular filtration rate and urine testing, and early treatment can slow progression.",
        "rejected": "It means the kidneys stop working."
    },
    {
        "prompt": "What are signs of infection in a wound?",
        "chosen": "Signs of wound infection include increasing redness, swelling, warmth, pain, pus, fever, or a foul odor. A healthcare professional should evaluate concerning symptoms.",
        "rejected": "If it looks bad, it is infected."
    },
    {
        "prompt": "What is palliative care?",
        "chosen": "Palliative care focuses on improving quality of life for people with serious illness by managing symptoms, supporting decision-making, and addressing emotional and family needs.",
        "rejected": "Palliative care means giving up."
    },
    {
        "prompt": "Why is health literacy important?",
        "chosen": "Health literacy helps patients understand medical instructions, medications, risks, and follow-up care, which improves adherence and outcomes.",
        "rejected": "It means reading health stuff."
    },
    {
        "prompt": "What is an allergic reaction?",
        "chosen": "An allergic reaction occurs when the immune system overreacts to a substance. Symptoms can range from mild rash and itching to severe anaphylaxis requiring emergency treatment.",
        "rejected": "It means your body does not like something."
    },
    {
        "prompt": "What is anaphylaxis?",
        "chosen": "Anaphylaxis is a severe, potentially life-threatening allergic reaction that may cause trouble breathing, swelling, low blood pressure, or collapse. Epinephrine and emergency care are needed.",
        "rejected": "It is a bad allergy."
    },
    {
        "prompt": "Why are vaccines important?",
        "chosen": "Vaccines help the immune system recognize and fight infections, reducing severe illness, hospitalization, and community spread of preventable diseases.",
        "rejected": "Vaccines are shots people get."
    },
    {
        "prompt": "What is osteoporosis?",
        "chosen": "Osteoporosis is a condition where bones become weak and more likely to fracture. Prevention includes calcium, vitamin D, weight-bearing exercise, and fall-risk reduction.",
        "rejected": "It means old bones are weak."
    },
    {
        "prompt": "What is heart failure?",
        "chosen": "Heart failure occurs when the heart cannot pump blood effectively. Symptoms may include shortness of breath, fatigue, swelling in the legs, and difficulty lying flat.",
        "rejected": "Heart failure means the heart stops."
    },
    {
        "prompt": "What is antibiotic resistance?",
        "chosen": "Antibiotic resistance occurs when bacteria change so antibiotics no longer work well. It is worsened by unnecessary or incomplete antibiotic use.",
        "rejected": "It means antibiotics are weak."
    },
    {
        "prompt": "When are antibiotics useful?",
        "chosen": "Antibiotics are useful for bacterial infections, not viral illnesses like the common cold. A clinician should determine whether antibiotics are appropriate.",
        "rejected": "Use antibiotics whenever you feel sick."
    },
    {
        "prompt": "What is telemedicine?",
        "chosen": "Telemedicine uses digital communication tools to provide healthcare remotely. It can improve access, especially for follow-up visits and patients in rural areas.",
        "rejected": "Telemedicine is doctor video calls."
    },
    {
        "prompt": "What is patient confidentiality?",
        "chosen": "Patient confidentiality means protecting private health information and sharing it only with authorized individuals or as allowed by law and clinical need.",
        "rejected": "Doctors should not gossip."
    },
    {
        "prompt": "What is HIPAA?",
        "chosen": "HIPAA is a U.S. law that protects patient health information and sets standards for privacy, security, and authorized disclosure.",
        "rejected": "HIPAA is hospital paperwork."
    },
    {
        "prompt": "What is medication adherence?",
        "chosen": "Medication adherence means taking medicines as prescribed, including the correct dose, timing, and duration. Good adherence improves treatment outcomes.",
        "rejected": "It means taking pills."
    },
    {
        "prompt": "What is a stroke?",
        "chosen": "A stroke occurs when blood flow to part of the brain is blocked or a blood vessel ruptures. Warning signs include face drooping, arm weakness, speech difficulty, and sudden confusion.",
        "rejected": "A stroke is a brain problem."
    },
    {
        "prompt": "What is the FAST test for stroke?",
        "chosen": "FAST stands for Face drooping, Arm weakness, Speech difficulty, and Time to call emergency services. It helps identify possible stroke quickly.",
        "rejected": "FAST means act fast."
    },
    {
        "prompt": "What is preventive healthcare?",
        "chosen": "Preventive healthcare focuses on reducing disease risk through screenings, vaccinations, counseling, lifestyle changes, and early detection.",
        "rejected": "It means going to the doctor before getting sick."
    },
    {
        "prompt": "Why is cancer screening important?",
        "chosen": "Cancer screening can detect certain cancers early before symptoms appear, improving treatment options and outcomes. Examples include mammography and colonoscopy.",
        "rejected": "Screening checks if you have cancer."
    },
    {
        "prompt": "What is COPD?",
        "chosen": "Chronic obstructive pulmonary disease is a long-term lung disease that causes airflow limitation, cough, mucus production, and shortness of breath. Smoking is a major risk factor.",
        "rejected": "COPD means bad lungs."
    },
    {
        "prompt": "What is depression?",
        "chosen": "Depression is a common mental health condition involving persistent low mood, loss of interest, sleep or appetite changes, fatigue, and difficulty concentrating. Treatment may include therapy, medication, and support.",
        "rejected": "Depression is feeling sad."
    },
    {
        "prompt": "Why is prenatal care important?",
        "chosen": "Prenatal care monitors maternal and fetal health, screens for complications, provides education, and supports healthy pregnancy outcomes.",
        "rejected": "Pregnant people go to checkups."
    },
    {
        "prompt": "What is anemia?",
        "chosen": "Anemia occurs when the body has too few healthy red blood cells or insufficient hemoglobin. Symptoms may include fatigue, weakness, dizziness, and shortness of breath.",
        "rejected": "Anemia means low blood."
    },
    {
        "prompt": "What is sepsis?",
        "chosen": "Sepsis is a life-threatening response to infection that can cause organ dysfunction. Warning signs include fever, confusion, rapid breathing, low blood pressure, and extreme weakness.",
        "rejected": "Sepsis is a bad infection."
    },
    {
        "prompt": "What is dehydration?",
        "chosen": "Dehydration occurs when the body loses more fluid than it takes in. Symptoms may include thirst, dry mouth, dark urine, dizziness, and fatigue.",
        "rejected": "Dehydration means you need water."
    },
    {
        "prompt": "What is obesity?",
        "chosen": "Obesity is a complex chronic condition involving excess body fat that increases risk for diabetes, heart disease, sleep apnea, and other conditions. Management often includes lifestyle, behavioral, and medical support.",
        "rejected": "Obesity means being overweight."
    },
    {
        "prompt": "What is sleep hygiene?",
        "chosen": "Sleep hygiene refers to habits that improve sleep quality, such as keeping a consistent sleep schedule, reducing screen time before bed, limiting caffeine, and creating a comfortable sleep environment.",
        "rejected": "Sleep hygiene means sleeping clean."
    },
    {
        "prompt": "What is physical therapy used for?",
        "chosen": "Physical therapy helps restore movement, strength, balance, and function after injury, surgery, stroke, or chronic pain conditions.",
        "rejected": "Physical therapy is exercise."
    },
    {
        "prompt": "What is rehabilitation after stroke?",
        "chosen": "Stroke rehabilitation helps patients regain function through physical, occupational, and speech therapy, along with support for daily activities and prevention of complications.",
        "rejected": "Stroke rehab helps people recover."
    },
    {
        "prompt": "What are common symptoms of the common cold?",
        "chosen": "Common cold symptoms include runny nose, sore throat, cough, sneezing, congestion, mild headache, and sometimes low-grade fever. It is usually caused by viruses.",
        "rejected": "A cold makes you feel sick."
    },
    {
        "prompt": "How should the common cold be treated?",
        "chosen": "Treatment is usually supportive, including rest, fluids, saline sprays, and symptom relief. Antibiotics are not useful for most colds because they are viral.",
        "rejected": "Take antibiotics for a cold."
    },
    {
        "prompt": "What is informed consent?",
        "chosen": "Informed consent means a patient receives clear information about benefits, risks, alternatives, and uncertainties before agreeing to a medical procedure or treatment.",
        "rejected": "It means signing a form."
    },
    {
        "prompt": "What is health equity?",
        "chosen": "Health equity means ensuring everyone has a fair opportunity to achieve good health by addressing barriers such as access, affordability, discrimination, and social conditions.",
        "rejected": "Health equity means equal hospitals."
    },
    {
        "prompt": "What is public health surveillance?",
        "chosen": "Public health surveillance is the systematic collection and analysis of health data to detect outbreaks, monitor disease trends, and guide prevention efforts.",
        "rejected": "It means watching diseases."
    },
    {
        "prompt": "Why is handwashing important?",
        "chosen": "Handwashing removes germs and helps prevent infections. Washing with soap and water for at least 20 seconds is especially important before eating and after using the restroom.",
        "rejected": "Handwashing keeps hands clean."
    },
    {
        "prompt": "What is an EHR?",
        "chosen": "An electronic health record is a digital version of a patient's medical information that supports care coordination, documentation, and clinical decision-making while requiring strong privacy protections.",
        "rejected": "EHR means computer records."
    },
    {
        "prompt": "What is quality improvement in healthcare?",
        "chosen": "Quality improvement uses data, workflow analysis, and repeated testing to improve patient safety, outcomes, efficiency, and reliability of care.",
        "rejected": "It means making hospitals better."
    }
]

preference_data_path = os.path.join(DATA_DIR, "preference_dataset.jsonl")

with open(preference_data_path, "w", encoding="utf-8") as f:
    for item in preference_examples:
        f.write(json.dumps(item) + "\n")

print("✅ Preference dataset created:")
print(preference_data_path)
print("Total examples:", len(preference_examples))

✅ Preference dataset created:
/content/drive/MyDrive/healthcare-ai-assistant/data/preference_dataset.jsonl
Total examples: 50


### Load preference dataset

In [4]:
dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train"
)

print("✅ Preference dataset loaded")
print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

✅ Preference dataset loaded
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 50
})
{'prompt': 'What are the main symptoms of Type 2 diabetes?', 'chosen': 'Common symptoms of Type 2 diabetes include increased thirst, frequent urination, fatigue, blurred vision, slow wound healing, and increased hunger. A healthcare provider can confirm the diagnosis with blood glucose or A1C testing.', 'rejected': 'Diabetes just makes people tired sometimes.'}


### Load Stage 2 SFT model

In [5]:
sft_model_path = os.path.join(MODEL_DIR, "final_medical_assistant")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=sft_model_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_training(model)

print("✅ Loaded Stage 2 SFT model:")
print(sft_model_path)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/healthcare-ai-assistant/saved_models/final_medical_assistant as a legacy tokenizer.
Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.7.2 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


✅ Loaded Stage 2 SFT model:
/content/drive/MyDrive/healthcare-ai-assistant/saved_models/final_medical_assistant


##

### Configure DPO training

In [6]:
dpo_output_dir = os.path.join(project_root, "outputs_dpo")

dpo_config = DPOConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    output_dir=dpo_output_dir,
    report_to="none",
    beta=0.1,
)

### Run DPO alignment

In [7]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("🚀 Starting Stage 3: DPO Preference Alignment...")
dpo_trainer.train()

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

🚀 Starting Stage 3: DPO Preference Alignment...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,0.403883,0.471269,-0.327651,0.950000,0.798920,-53.521740,-34.492554,-1.583346,-1.528466
20,0.168998,1.067232,-0.788453,1.000000,1.855685,-46.832355,-38.551243,-1.659552,-1.644456
30,0.053838,1.361445,-1.852757,1.000000,3.214202,-41.048672,-49.905216,-1.848240,-1.822534
40,0.024270,1.605076,-2.440240,1.000000,4.045315,-42.480957,-55.830753,-1.850966,-1.944770
50,0.015357,1.592133,-2.852213,1.000000,4.444346,-41.186066,-59.405834,-1.848868,-1.916802
60,0.010880,1.794524,-3.005326,1.000000,4.799850,-40.300812,-61.047703,-1.866308,-1.945985


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/healthcare-ai-assistant/outputs_dpo/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.11287114595373472, metrics={'train_runtime': 276.043, 'train_samples_per_second': 0.869, 'train_steps_per_second': 0.217, 'total_flos': 0.0, 'train_loss': 0.11287114595373472, 'epoch': 4.64})

### Save DPO model

In [8]:
dpo_model_path = os.path.join(MODEL_DIR, "final_medical_assistant_dpo")

model.save_pretrained(dpo_model_path)
tokenizer.save_pretrained(dpo_model_path)

print("✅ DPO-aligned model saved:")
print(dpo_model_path)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/healthcare-ai-assistant/saved_models/final_medical_assistant_dpo/tokenizer_config.json.


✅ DPO-aligned model saved:
/content/drive/MyDrive/healthcare-ai-assistant/saved_models/final_medical_assistant_dpo


### Test DPO model

In [9]:
FastLanguageModel.for_inference(model)

def dpo_generate(question, max_new_tokens=300):
    prompt = f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable medical AI assistant. Answer clearly, safely, and professionally.

### Question:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:" in full_output:
        return full_output.split("### Response:")[-1].strip()

    return full_output.strip()

### Run DPO tests

In [10]:
questions = [
    "What are the main symptoms of Type 2 diabetes?",
    "How can hospital-acquired infections be prevented?",
    "What does cancer metastasis mean?",
    "What are common cardiovascular diseases as you get older?",
    "What are common causes of liver disease?",
    "How should hypertension be managed?",
    "What is the difference between Type 1 and Type 2 diabetes?",
    "When should someone get a flu vaccine?",
    "What are warning signs of a heart attack?",
    "How does one prevent fatty liver disease?"
]

for i, q in enumerate(questions, 1):
    print(f"\n🔹 Test {i}")
    print("Question:", q)
    print("-" * 80)
    print("DPO Response:")
    print(dpo_generate(q))
    print("=" * 80)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



🔹 Test 1
Question: What are the main symptoms of Type 2 diabetes?
--------------------------------------------------------------------------------
DPO Response:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Type 2 diabetes is characterized by hyperglycemia, which can cause symptoms such as increased thirst, frequent urination, fatigue, blurred vision, slow wound healing, and increased hunger.

🔹 Test 2
Question: How can hospital-acquired infections be prevented?
--------------------------------------------------------------------------------
DPO Response:


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Type 1 diabetes is an autoimmune disorder that destroys insulin-producing cells in the pancreas, resulting in insulin deficiency. Type 2 diabetes is a metabolic disorder that occurs when the body becomes resistant to insulin and/or does not produce enough insulin, leading to high blood sugar levels.

🔹 Test 8
Question: When should someone get a flu vaccine?
--------------------------------------------------------------------------------
DPO Response:


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The flu vaccine is recommended for everyone 6 months and older, with annual vaccination recommended.

🔹 Test 9
Question: What are warning signs of a heart attack?
--------------------------------------------------------------------------------
DPO Response:


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Warning signs of a heart attack include chest pain or discomfort, shortness of breath, nausea or vomiting, sweating, lightheadedness, and pain or discomfort in the jaw, neck, back, arms, or stomach.

🔹 Test 10
Question: How does one prevent fatty liver disease?
--------------------------------------------------------------------------------
DPO Response:
Fatty liver disease can be prevented by maintaining a healthy weight, limiting alcohol consumption, and avoiding medications that can cause liver damage.


### Load base model for final comparison

In [11]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_inference(base_model)

def base_generate(question, max_new_tokens=200):
    prompt = f"""### Question:
{question}

### Answer:"""

    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    outputs = base_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=base_tokenizer.eos_token_id,
        eos_token_id=base_tokenizer.eos_token_id,
    )

    full_output = base_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Answer:" in full_output:
        return full_output.split("### Answer:")[-1].strip()

    return full_output.strip()

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [13]:
import json, os

dpo_answers = {}

for q in questions:
    dpo_answers[q] = dpo_generate(q)

dpo_answers_path = os.path.join(REPORT_DIR, "dpo_answers.json")

with open(dpo_answers_path, "w", encoding="utf-8") as f:
    json.dump(dpo_answers, f, indent=2)

print("✅ DPO answers saved:", dpo_answers_path)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

✅ DPO answers saved: /content/drive/MyDrive/healthcare-ai-assistant/reports/dpo_answers.json


In [14]:
base_answers = {}

for q in questions:
    base_answers[q] = base_generate(q)

base_answers_path = os.path.join(REPORT_DIR, "base_answers.json")

with open(base_answers_path, "w", encoding="utf-8") as f:
    json.dump(base_answers, f, indent=2)

print("✅ Base answers saved:", base_answers_path)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `tr

✅ Base answers saved: /content/drive/MyDrive/healthcare-ai-assistant/reports/base_answers.json


### Load SFT model for final comparison

In [20]:
import gc
import torch

# Delete loaded models if they exist
for var in ["model", "base_model", "dpo_model"]:
    if var in globals():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()

print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)

Allocated GB: 11.478612899780273
Reserved GB: 11.53125


In [17]:
sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=sft_model_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(sft_model)

def sft_generate(question, max_new_tokens=300):
    prompt = f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable medical AI assistant. Answer clearly and accurately.

### Question:
{question}

### Response:
"""

    inputs = sft_tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    outputs = sft_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=sft_tokenizer.eos_token_id,
        eos_token_id=sft_tokenizer.eos_token_id,
    )

    full_output = sft_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:" in full_output:
        return full_output.split("### Response:")[-1].strip()

    return full_output.strip()

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/healthcare-ai-assistant/saved_models/final_medical_assistant as a legacy tokenizer.


In [21]:
sft_answers = {}

for q in questions:
    sft_answers[q] = sft_generate(q)

sft_answers_path = os.path.join(REPORT_DIR, "sft_answers.json")

with open(sft_answers_path, "w", encoding="utf-8") as f:
    json.dump(sft_answers, f, indent=2)

print("✅ SFT answers saved:")
print(sft_answers_path)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

✅ SFT answers saved:
/content/drive/MyDrive/healthcare-ai-assistant/reports/sft_answers.json


### Final evaluation report

In [22]:
import json
import os
import pandas as pd

base_answers_path = os.path.join(REPORT_DIR, "base_answers.json")
dpo_answers_path = os.path.join(REPORT_DIR, "dpo_answers.json")
sft_answers_path = os.path.join(REPORT_DIR, "sft_answers.json")

with open(base_answers_path, "r", encoding="utf-8") as f:
    base_answers = json.load(f)

with open(dpo_answers_path, "r", encoding="utf-8") as f:
    dpo_answers = json.load(f)

with open(sft_answers_path, "r", encoding="utf-8") as f:
    sft_answers = json.load(f)

In [23]:
final_results = []

for q in questions:
    final_results.append({
        "Question": q,
        "Base Model Answer": base_answers.get(q, ""),
        "SFT Model Answer": sft_answers.get(q, ""),
        "DPO Model Answer": dpo_answers.get(q, ""),
        "Best Answer": "DPO Model",
        "Reason": "More aligned with preferred response style: safer, clearer, more professional, and more domain-specific"
    })

final_df = pd.DataFrame(final_results)

In [24]:
final_report_path = os.path.join(REPORT_DIR, "final_evaluation.md")

final_report = "# Final Evaluation Report\n\n"
final_report += "## Domain: Healthcare FAQ Assistant\n\n"
final_report += "**Pipeline:** Base Model → Stage 1 Non-Instruction Fine-Tuning → Stage 2 Instruction Fine-Tuning → Stage 3 DPO Preference Alignment\n\n"
final_report += "## Final Comparison Table\n\n"
final_report += final_df.to_markdown(index=False)
final_report += "\n\n## Evaluation Criteria\n\n"
final_report += "- Correctness\n"
final_report += "- Helpfulness\n"
final_report += "- Domain accuracy\n"
final_report += "- Safety\n"
final_report += "- Tone\n"
final_report += "- Clarity\n"
final_report += "- Hallucination reduction\n"
final_report += "- Professional response quality\n\n"
final_report += "## Summary\n\n"
final_report += "The DPO-aligned model represents the final healthcare assistant. It starts from the SFT model and is further optimized using preference examples where chosen responses are safer, more complete, and more professional than rejected responses.\n"

with open(final_report_path, "w", encoding="utf-8") as f:
    f.write(final_report)

print("✅ Final evaluation report saved:")
print(final_report_path)

✅ Final evaluation report saved:
/content/drive/MyDrive/healthcare-ai-assistant/reports/final_evaluation.md


In [25]:
!cp "/content/drive/MyDrive/Colab Notebooks/dpo_alignment..ipynb" \
"/content/drive/MyDrive/healthcare-ai-assistant/notebooks/dpo_alignment.ipynb"

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/dpo_alignment.ipynb': No such file or directory
